# Phase 6 — TF-IDF Feature Extraction Notebook

**Project:** AI-Powered Customer Feedback Intelligence System  
**Goal:** Convert clean review text into numerical TF-IDF feature matrices for machine learning algorithms.

### Objectives:
1. Load `X_train` (314,863 samples) and `X_test` (78,716 samples)
2. Explain TF-IDF Theory (Term Frequency & Inverse Document Frequency)
3. Configure `TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)`
4. Fit vectorizer strictly on `X_train` (`fit_transform`)
5. Transform `X_test` (`transform`) to prevent Data Leakage
6. Inspect Matrix Shapes, Sparsity, and Learned Features
7. Save fitted vectorizer to `models/tfidf_vectorizer.pkl`

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Load cleaned dataset
df = pd.read_csv('../data/processed/reviews_cleaned.csv')
df['clean_text'] = df['clean_text'].fillna('')

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['Sentiment'], test_size=0.20, random_state=42, stratify=df['Sentiment']
)

print(f'X_train samples: {len(X_train):,}')
print(f'X_test samples : {len(X_test):,}')

## 2. TF-IDF Concept & Formula
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$
* **Term Frequency (TF)**: Relative frequency of word $t$ in document $d$.
* **Inverse Document Frequency (IDF)**: Penalizes words that appear across almost all documents (e.g. `the`, `is`, `product`) and boosts domain-specific sentiment terms (`excellent`, `terrible`, `not good`).

## 3. Fit Vectorizer (Train Only) & Transform Test
> [!IMPORTANT]
> **DATA LEAKAGE SAFETY**: Vectorizer is fitted using `fit_transform()` ONLY on `X_train`. `X_test` is transformed using `transform()`.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'X_train_tfidf shape: {X_train_tfidf.shape}')
print(f'X_test_tfidf shape : {X_test_tfidf.shape}')

## 4. Feature Inspection & Sparse Matrix Check

In [ ]:
features = tfidf.get_feature_names_out()
print(f'Total Features Learned: {len(features):,}')
print('Sample Unigrams & Bigrams:', list(features[5000:5015]))

# Save vectorizer
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')
print('Vectorizer saved to models/tfidf_vectorizer.pkl')